# 4주차 강사용 Notebook — 결측값과 이상한 데이터 정리하기

학생용 Notebook과 동일한 흐름이며, 예상 결과와 설명 포인트를 함께 담았습니다.

## 1단계. Pandas 불러오고 데이터 읽기

In [1]:
import pandas as pd
df = pd.read_csv("../../data/weekly/week04/week04_dirty_process_data.csv")
df.head()

,측정시간,로트번호,설비번호,공정명,온도_섭씨,압력_Pa,가스유량_slm,처리시간_sec,합격여부
0,2024-03-01 23:00,LOT-0336,EQ-01,식각,297.5,1010.0,49.92,123.1,1.0
1,2024-03-02 08:00,LOT-0281,EQ-02,포토,299.3,1012.1,47.91,114.6,1.0
2,2024-03-02 12:00,LOT-0304,EQ-01,세정,297.5,1008.9,51.20,115.3,1.0
3,2024-03-02 16:00,LOT-0254,EQ-02,식각,294.8,1024.5,51.29,110.5,-1.0
4,2024-03-03 02:00,LOT-0020,EQ-04,산화,302.6,997.3,45.75,117.4,1.0


## 2단계. 데이터 크기 확인하기
**설명 포인트**: 데이터 설명서에는 '약 110행'이라고 적혀 있지만 실제로는 113행이다. 실습 전에 실제 shape를 확인하는 습관을 강조한다.

In [2]:
print(df.shape)

(113, 9)


**예상 결과**: `(113, 9)`

## 3단계. 결측값 개수 세기
**어려워할 부분**: `isna()`만 실행하면 True/False 표가 나온다는 것을 모를 수 있음. 반드시 `.sum()`을 붙여야 개수가 나온다는 점을 강조.

In [3]:
print(df.isna().sum())

측정시간        0
로트번호        0
설비번호        0
공정명         0
온도_섭씨       4
압력_Pa       3
가스유량_slm    1
처리시간_sec    0
합격여부        2
dtype: int64


**예상 결과**: 온도_섭씨 4, 압력_Pa 3, 가스유량_slm 1, 합격여부 2 (나머지 열은 0).

## 4단계. 결측값이 있는 행 직접 보기

In [4]:
df[df.isna().any(axis=1)]

,측정시간,로트번호,설비번호,공정명,온도_섭씨,압력_Pa,가스유량_slm,처리시간_sec,합격여부
6,2024-03-04 17:00,LOT-0287,EQ-01,증착,298.6,1021.4,NaN,131.8,1.0
25,2024-03-10 05:00,LOT-0168,EQ-04,중착,NaN,1006.2,53.36,115.0,1.0
29,2024-03-11 19:00,LOT-0180,EQ-02,세정,295.5,NaN,49.99,117.6,1.0
34,2024-03-13 12:00,LOT-0396,EQ-04,산화,298.7,NaN,54.86,113.7,1.0
44,2024-03-17 05:00,LOT-0232,EQ-02,세정,303.3,1006.2,49.12,117.1,NaN
76,2024-03-26 07:00,LOT-0039,EQ-02,산화,NaN,1022.4,45.96,112.8,1.0
83,2024-03-28 13:00,LOT-0219,EQ-04,식각,303.6,NaN,49.71,119.7,NaN
96,2024-04-03 12:00,LOT-0201,EQ-01,산화,NaN,1007.4,50.89,124.6,1.0
106,2024/03/05,LOT-0383,EQ-04,증착,NaN,1011.0,47.50,116.6,1.0


**예상 결과**: 9개 행이 표시됨(결측이 2개 겹치는 행이 1개 있어 10건의 결측이 9개 행에 분포).

## 5단계. 결측값 제거 vs 대체 비교
**설명 포인트**: 온도처럼 이상값(500℃대)이 섞여 있는 열은 평균보다 중앙값을 쓰는 이유를 실제 평균/중앙값 값을 비교해서 보여준다.

In [5]:
df_dropped = df.dropna()
print("제거 후 행 개수:", df_dropped.shape[0])

mean_temp = df["온도_섭씨"].mean()
median_temp = df["온도_섭씨"].median()
print("온도_섭씨 평균:", round(mean_temp, 2), "/ 중앙값:", median_temp)

제거 후 행 개수: 104
온도_섭씨 평균: 303.86 / 중앙값: 299.3


**예상 결과**: 제거 후 104행. 평균(약 303.9)이 중앙값(약 299.2)보다 훨씬 높다 — 500℃대 이상값 2건이 평균을 끌어올렸기 때문임을 학생들과 함께 확인한다.

## 6단계. 중복 행 찾고 정리하기

In [6]:
print("중복 행 개수:", df.duplicated().sum())

df_no_dup = df.drop_duplicates()
print("중복 제거 후 행 개수:", df_no_dup.shape[0])

중복 행 개수: 3
중복 제거 후 행 개수: 110


**예상 결과**: 중복 3건 → 제거 후 110행.

## 7단계. 이상값 찾기
**질문할 내용**: "압력은 왜 음수가 될 수 없을까요?" (물리적으로 압력은 항상 0 이상).

In [7]:
print(df[df["압력_Pa"] < 0])
print(df[df["온도_섭씨"] > 400])
print(df[(df["가스유량_slm"] < 0) | (df["가스유량_slm"] > 200)])

                측정시간      로트번호   설비번호 공정명  온도_섭씨      압력_Pa  가스유량_slm  \
37  2024-03-13 16:00  LOT-0238  EQ-03  식각  308.4 -40.370466     50.85   
79  2024-03-26 21:00  LOT-0121  EQ-04  산화  300.9 -28.297835     44.92   

    처리시간_sec  합격여부  
37     126.0   1.0  
79     113.5   1.0  
                측정시간      로트번호   설비번호 공정명       온도_섭씨   압력_Pa  가스유량_slm  \
17  2024-03-07 23:00  LOT-0202  EQ-04  포토  520.879883  1017.4     51.59   
67  2024-03-24 08:00  LOT-0177  EQ-01  증착  519.854978  1029.9     48.75   

    처리시간_sec  합격여부  
17     119.6   1.0  
67     123.7   1.0  
                측정시간      로트번호   설비번호 공정명  온도_섭씨   압력_Pa  가스유량_slm  처리시간_sec  \
65  2024-03-23 21:00  LOT-0271  EQ-01  증착  307.8  1005.4     250.0     125.9   
86  2024-03-28 19:00  LOT-0081  EQ-01  세정  295.5  1015.4     250.0     121.7   

    합격여부  
65   1.0  
86   1.0  


**예상 결과**: 음수 압력 2건, 극단 고온 2건, 비정상 가스유량 2건(모두 250.0).

## 8단계. 공정명 오타 확인 및 통일
**실습 시간 조절**: 시간이 부족하면 이 단계는 도전 실습으로 넘겨도 된다.

In [8]:
print(df["공정명"].unique())

['식각' '포토' '세정' '산화' '증착' '중착' '포토공정' '식각 ']


**예상 결과**: `['식각' '포토' '세정' '산화' '증착' '중착' '포토공정' '식각 ']` — 오타 3건(중착, 포토공정, 끝에 공백이 붙은 '식각 ')이 정상 공정명과 섞여 있음을 확인.

## 9단계. 전체 정제 파이프라인 실행
중복 제거 → 공정명 오타 통일 → 이상값 제거 → 합격여부 결측 제거 → 남은 결측 중앙값 대체 순서로 진행한다.

In [9]:
df_clean = df.drop_duplicates().copy()
df_clean["공정명"] = df_clean["공정명"].str.strip().replace({"중착": "증착", "포토공정": "포토"})

bad_temp = df_clean["온도_섭씨"] > 400
bad_pressure = df_clean["압력_Pa"] < 0
bad_gas = (df_clean["가스유량_slm"] < 0) | (df_clean["가스유량_slm"] > 200)
df_clean = df_clean[~(bad_temp | bad_pressure | bad_gas)].copy()

df_clean = df_clean.dropna(subset=["합격여부"])

df_clean = df_clean.fillna({
    "온도_섭씨": df_clean["온도_섭씨"].median(),
    "압력_Pa": df_clean["압력_Pa"].median(),
    "가스유량_slm": df_clean["가스유량_slm"].median(),
})

print("정제 후 행 개수:", df_clean.shape[0])
print("정제 후 결측값 합계:", df_clean.isna().sum().sum())
print("정제 후 공정명 목록:", sorted(df_clean["공정명"].unique()))

정제 후 행 개수: 102
정제 후 결측값 합계: 0
정제 후 공정명 목록: ['산화', '세정', '식각', '증착', '포토']


**예상 결과**: 113행 → 110행(중복 제거) → 104행(이상값 제거) → 102행(합격여부 결측 제거), 최종 102행에 결측값 0건, 공정명은 5종(증착/포토/세정/산화/식각)으로 통일됨.

## 10단계. 정제 전후 비교

In [10]:
mean_before = df["온도_섭씨"].mean()
mean_after = df_clean["온도_섭씨"].mean()
print("정제 전 평균 온도:", round(mean_before, 2))
print("정제 후 평균 온도:", round(mean_after, 2))
print("평균 차이:", round(mean_before - mean_after, 2))
print("행 개수:", df.shape[0], "->", df_clean.shape[0])

정제 전 평균 온도: 303.86
정제 후 평균 온도: 299.69
평균 차이: 4.18
행 개수: 113 -> 102


**예상 결과**: 정제 전 약 303.86℃, 정제 후 약 299.69℃, 차이 약 4.18℃. 정제 전 평균이 정상 범위(295~305℃) 상단에 걸려 있는 것처럼 보이지만, 실제로는 500℃대 이상값 2건이 평균을 끌어올린 결과라는 점을 강조한다.

## 오류 대처 방법
- `KeyError` 발생 시: `df.columns`로 정확한 열 이름을 확인.
- `fillna()`가 적용되지 않는 경우: `df["열이름"] = df["열이름"].fillna(값)`처럼 재할당했는지 확인.
- `SettingWithCopyWarning` 발생 시: `.copy()`를 붙여 명시적으로 복사본을 만들었는지 확인.

## 확장 실습(빠른 학습자용)
`df_clean.describe()`로 정제된 데이터의 요약 통계를 살펴보고, 정상 범위(295~305℃, 995~1025 Pa)와 비교해본다.

In [11]:
df_clean.describe()

,온도_섭씨,압력_Pa,가스유량_slm,처리시간_sec,합격여부
count,102.000000,102.000000,102.000000,102.000000,102.000000
mean,299.685294,1012.364706,50.046275,118.777451,0.725490
std,4.113860,7.945886,1.993305,5.697245,0.691631
min,287.000000,995.100000,44.800000,102.800000,-1.000000
25%,297.425000,1006.350000,48.882500,115.000000,1.000000
50%,299.200000,1012.250000,50.010000,118.550000,1.000000
75%,301.975000,1018.275000,51.422500,122.600000,1.000000
max,310.500000,1029.900000,54.860000,131.800000,1.000000
